# Multi-Robot MjSpec Helper Class
- internally generate mjSpec object
- Attach Frame & Add Robot
    - Receive: base transform(position & rotation), Robot path
    - Prefix or suffix

In [1]:
""" IMPORTS """

import os
import sys
import numpy as np
import time
import xml.etree.ElementTree as ET
from lxml import etree

import mujoco
sys.path.append(os.path.abspath('../'))
from pp_base_mujoco.VIEWER import *
from pp_base_mujoco.UTILS import *

In [2]:
class MJSPECHELPER:
    def __init__(
            self,
            xml_path = "../asset/floor_white_gray.xml"
            ): 
        self.spec = mujoco.MjSpec.from_file(xml_path)

    def add_robot(
            self,
            path,
            body_name = "base_link",
            p=(0, 0, 0),
            r=(0, 0, 0),
            prefix = "",
            suffix = ""
    ):
        """ 
        Add robot to MjSpec
        Args:
            - path: path to robot xml file
            - p: position of the robot
            - r: rotation of the robot
            - prefix: prefix for the robot name
            - suffix: suffix for the robot name
        """

        frame = self.spec.worldbody.add_frame(pos=p, euler = r)
        robot_spec = mujoco.MjSpec.from_file(path)
        frame.attach_body(
            robot_spec.body(body_name), # assume that each xml has worldbody element 
            prefix=prefix,
            suffix=suffix
            )
    
    def add_geom(
            self,
            type,
            size,
            name = "",
            freejoint=False,
            p=(0, 0, 0),
            r=(0, 0, 0),
            rgba=(0.3, 0.3, 0.3, 0.5),
            group=0,
            friction=(1.0, 0.005, 0.0001),
            mass=1.0 # 1kg
    ):
        """ 
        Add geometry to MjSpec, with body encapsulation
        Args:
            - type: type of the geometry (e.g., "box", "sphere", "cylinder")
            - size: size of the geometry (e.g., for box: [length, width, height], for sphere: [radius], for cylinder: [radius, height])
            - freejoint: whether the geometry is freejoint
            - p: position of the geometry
            - r: rotation of the geometry
            - name: name of the geometry
            - rgba: color and transparency of the geometry (optional)
        """
        body = self.spec.worldbody.add_body(
            name=name,
            pos=p,
            euler=r
        )
        if freejoint:
            body.add_freejoint()
        if type == "box":
            body.add_geom(
                type       = mujoco.mjtGeom.mjGEOM_BOX,
                size       = size,
                pos        = np.zeros((3)),
                euler      = [0, 0, 0],
                rgba       = rgba
            )
        elif type == "sphere":
            body.add_geom(
                type       = mujoco.mjtGeom.mjGEOM_SPHERE,
                size       = size,
                pos        = np.zeros((3)),
                euler      = [0, 0, 0],
                rgba       = rgba
            )
        elif type == "cylinder":
            body.add_geom(
                type       = mujoco.mjtGeom.mjGEOM_CYLINDER,
                size       = size,
                pos        = np.zeros((3)),
                euler      = [0, 0, 0],
                rgba       = rgba
            )
        else:
            raise ValueError("Unsupported geometry type: {}".format(type))
        
    def compile(self):
        """ Compile the MjSpec to check for errors and prepare for simulation """
        model = self.spec.compile()
        data = mujoco.MjData(model)
        return model, data
    
    def print_xml(self):
        """ Print the XML representation of the MjSpec """
        xml_str = self.spec.to_xml()
        if isinstance(xml_str, ET.Element):
            rough_string = ET.tostring(xml_str, encoding='unicode')
        else:
            rough_string = xml_str

        parser = etree.XMLParser(remove_blank_text=True)
        tree = etree.fromstring(rough_string, parser=parser)
        pretty_xml = etree.tostring(tree, pretty_print=True, encoding='unicode')
        print(pretty_xml)

#### 2. Main Code - Box

In [3]:
spec_helper = MJSPECHELPER()
spec_helper.add_geom(
    type="box",
    size=[0.25, 0.25, 0.25],
    freejoint=True,
    p=(0, 0, 1.0),
    r=(0, 0, 0),
    name="box1"
)
model, data = spec_helper.compile()

In [4]:
""" MAIN LOOP """

# create python viewer object
viewer = MUJOCOGLVIEWER(model, data)

# data reset
mujoco.mj_resetData(model, data)

while viewer.is_alive():
        mujoco.mj_step(model, data)
        viewer.render()

# close
viewer.close()
del(viewer)

2026-05-09 13:11:12.569 python[67041:8720894] TSM AdjustCapsLockLEDForKeyTransitionHandling - _ISSetPhysicalKeyboardCapsLockLED Inhibit


#### 2-2. Main Code - Arm and Box

In [5]:
del(spec_helper)

In [6]:
spec_helper = MJSPECHELPER()
spec_helper.add_robot(
    path="../asset/panda/panda.xml",
    body_name="base",
    p=(0, 0.5, 0),
    r=(0, 0, -90),
    prefix="",
    suffix="_left"
)
spec_helper.add_robot(
    path="../asset/panda/panda.xml",
    body_name="base",
    p=(0, -0.5, 0),
    r=(0, 0, 90),
    prefix="",
    suffix="_right"
)
model, data = spec_helper.compile()

In [7]:
""" MAIN LOOP """

# create python viewer object
viewer = MUJOCOGLVIEWER(model, data)

# data reset
mujoco.mj_resetData(model, data)

while viewer.is_alive():
        mujoco.mj_step(model, data)
        viewer.render()

# close
viewer.close()
del(viewer)

In [8]:
""" CHECK BODY, JOINT NAMES """

body_names = get_body_names(model, data)
joint_names = get_joint_names(model, data)

print("Body Names: ", body_names)
print("Joint Names: ", joint_names)

Body Names:  ['world', 'base_left', 'link0_left', 'link1_left', 'link2_left', 'link3_left', 'link4_left', 'link5_left', 'link6_left', 'link7_left', 'right_hand_left', 'base_right', 'link0_right', 'link1_right', 'link2_right', 'link3_right', 'link4_right', 'link5_right', 'link6_right', 'link7_right', 'right_hand_right']
Joint Names:  ['joint1_left', 'joint2_left', 'joint3_left', 'joint4_left', 'joint5_left', 'joint6_left', 'joint7_left', 'joint1_right', 'joint2_right', 'joint3_right', 'joint4_right', 'joint5_right', 'joint6_right', 'joint7_right']
